In [1]:
import torch
from torch import nn

In [8]:
inputs = torch.Tensor([[[0.2, 0.1, 0.3], [0.5, 0.1, 0.1]]])
B, S, E = inputs.size() # B = Batch dimension. Permite paralelização, treino +rápido
inputs = inputs.reshape(S, B, E)
inputs.size()

torch.Size([2, 1, 3])

In [9]:
inputs

tensor([[[0.2000, 0.1000, 0.3000]],

        [[0.5000, 0.1000, 0.1000]]])

Com a dimensão Batch, não vamos usar LayerNorm() apenas para a última camada, mas para a última camada distribuída em Batches.

In [ ]:
parameter_shape = inputs.size()[-2:] # B, E
gamma = nn.Parameter(torch.ones(parameter_shape))
beta = nn.Parameter(torch.ones(parameter_shape))
gamma.size(), beta.size()

(torch.Size([1, 3]), torch.Size([1, 3]))

In [ ]:
dims = [-(i + 1) for i in range(len(parameter_shape))]
dims

[-1, -2]

In [10]:
mean = inputs.mean(dim=dims, keepdim=True)
mean.size()

torch.Size([2, 1, 1])

In [11]:
mean

tensor([[[0.2000]],

        [[0.2333]]])

In [ ]:
var = ((inputs - mean) ** 2).mean(dim=dims, keepdim=True)
epsilon = 1e-5
std = (var + epsilon).sqrt() # epsilon garante que std não seja zero
std

tensor([[[0.0817]],

        [[0.1886]]])

In [13]:
y = (inputs - mean) / std
y

tensor([[[ 0.0000, -1.2238,  1.2238]],

        [[ 1.4140, -0.7070, -0.7070]]])

In [14]:
out = gamma * y + beta
out

tensor([[[ 1.0000, -0.2238,  2.2238]],

        [[ 2.4140,  0.2930,  0.2930]]], grad_fn=<AddBackward0>)

Note que temos um adiciona grad_fn. Isso indica que temos parâmetros aprendíveis ($\gamma$, $\beta$).

# Class

In [ ]:
import torch
from torch import nn

class LayerNormalization():
    def __init__(self, parameters_shape, eps=1e-5):
        self.parameters_shape=parameters_shape
        self.eps=eps
        self.gamma = nn.Parameter(torch.ones(parameters_shape))
        self.beta =  nn.Parameter(torch.zeros(parameters_shape))

    def forward(self, input):
        dims = [-(i + 1) for i in range(len(self.parameters_shape))]
        mean = inputs.mean(dim=dims, keepdim=True)
        print(f"Mean \n ({mean.size()}): \n {mean}")
        var = ((inputs - mean) ** 2).mean(dim=dims, keepdim=True)
        std = (var + self.eps).sqrt()
        print(f"Standard Deviation \n ({std.size()}): \n {std}")
        y = (inputs - mean) / std
        print(f"y \n ({y.size()}) = \n {y}")
        out = self.gamma * y  + self.beta
        print(f"out \n ({out.size()}) = \n {out}")
        return out

In [16]:
batch_size = 3
sentence_length = 5
embedding_dim = 8 
inputs = torch.randn(sentence_length, batch_size, embedding_dim)

print(f"input \n ({inputs.size()}) = \n {inputs}")


input 
 (torch.Size([5, 3, 8])) = 
 tensor([[[ 1.5191,  0.8347, -1.0833, -1.6070, -0.6306, -0.1654,  0.1872,
           0.5540],
         [-0.2223,  0.9453,  1.3867,  0.1350,  1.0242,  0.0513,  1.1633,
          -0.1432],
         [-0.0592, -1.8812, -1.7871,  0.5793, -0.4652,  1.0843, -0.3827,
           0.0142]],

        [[-1.4196,  0.5918, -0.7412,  0.5385, -0.6404,  1.0447,  1.2818,
           0.4897],
         [ 0.7550,  0.8508, -0.5114,  0.4835,  0.1353, -0.7811,  0.1188,
           1.5438],
         [-0.0418,  0.2029,  0.1718, -1.8464, -0.8848,  0.2487,  1.2054,
           2.1397]],

        [[ 0.7897,  0.4087, -0.5086, -0.1609, -0.2790, -0.3866, -1.1756,
           0.1068],
         [-2.2211, -2.2935,  2.3961,  1.0517, -0.8057, -0.4425, -0.6126,
          -0.7643],
         [ 0.4783, -0.3888, -1.8758,  1.9151, -0.6084,  0.4710,  0.0172,
           0.4489]],

        [[-0.5051,  1.7252, -0.1249, -0.2682, -2.1481,  1.6810,  1.3565,
          -3.0160],
         [ 0.6891, -1.0187, 

In [18]:
layer_norm = LayerNormalization(inputs.size()[-1:])
out = layer_norm.forward(inputs)

Mean 
 (torch.Size([5, 3, 1])): 
 tensor([[[-0.0489],
         [ 0.5425],
         [-0.3622]],

        [[ 0.1432],
         [ 0.3243],
         [ 0.1494]],

        [[-0.1507],
         [-0.4615],
         [ 0.0572]],

        [[-0.1624],
         [-0.1852],
         [-0.1184]],

        [[-0.1350],
         [-0.2017],
         [ 0.1898]]])
Standard Deviation 
 (torch.Size([5, 3, 1])): 
 tensor([[[0.9688],
         [0.6077],
         [0.9722]],

        [[0.8962],
         [0.7048],
         [1.1280]],

        [[0.5606],
         [1.4638],
         [1.0206]],

        [[1.6388],
         [0.8181],
         [0.8463]],

        [[1.0918],
         [0.6866],
         [0.6647]]])
y 
 (torch.Size([5, 3, 8])) = 
 tensor([[[ 1.6186,  0.9121, -1.0677, -1.6083, -0.6005, -0.1202,  0.2437,
           0.6223],
         [-1.2585,  0.6626,  1.3891, -0.6705,  0.7926, -0.8082,  1.0214,
          -1.1283],
         [ 0.3116, -1.5625, -1.4656,  0.9685, -0.1060,  1.4879, -0.0211,
           0.3872]],



In [19]:
out[0].mean(), out[0].std()

(tensor(-1.9868e-08, grad_fn=<MeanBackward0>),
 tensor(1.0215, grad_fn=<StdBackward0>))

Veja que a média fica bem próxima de zero, além de que o grau de dispersão (standard deviation) é 1 (o que indica que estamos no intervalo [-1, 1])